## Dependencies

In [ ]:
# Install dependencies
%pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv unstructured beautifulsoup4 lxml python-docx pypandoc rank-bm25 -q

# Verify installation
import sys
print(f"Python version: {sys.version}")

# Verify key packages are installed
try:
    import langchain
    import langchain_core
    print(f"LangChain version: {langchain.__version__}")
    print("All core packages installed successfully!")
except ImportError as e:
    print(f"⚠️  Error: {e}")
    print("Please ensure all packages are installed correctly.")


## Setup & Imports

In [6]:
# Standard library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader, UnstructuredODTLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Modern LangChain agent imports (latest pattern from LangChain docs)
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Website scraping strainer
from bs4 import SoupStrainer

#download pypandoc
import pypandoc
pypandoc.download_pandoc()

# Load environment variables - check current directory first, then parent
current_dir = Path.cwd()
env_path = current_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"Loading .env from: {env_path}")
else:
    # Try parent directory
    parent_env = current_dir.parent / '.env'
    if parent_env.exists():
        load_dotenv(parent_env)
        print(f"Loading .env from: {parent_env}")
    else:
        # Fallback to default behavior (searches current and parent directories)
        load_dotenv()
        print("⚠️  No .env file found. Using default load_dotenv() search")

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please create a .env file in this directory with your OpenAI API key:")
    print(f"   {current_dir / '.env'}")
    print("\nFormat: OPENAI_API_KEY=sk-...")
else:
    # Validate API key format (basic check)
    if not api_key.startswith('sk-'):
        print("⚠️  WARNING: API key format looks incorrect. Should start with 'sk-'")
    else:
        print("OpenAI API key loaded successfully!")
        # Test the API key by making a simple call
        try:
            test_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=5)
            test_llm.invoke("Hi")
            print("API key validated successfully!")
        except Exception as e:
            print(f"❌ ERROR: API key validation failed!")
            print(f"   Error: {str(e)}")
            print("\n💡 Please check:")
            print("   1. Your API key is correct (get it from https://platform.openai.com/api-keys)")
            print("   2. You have sufficient credits in your OpenAI account")
            print("   3. The .env file is in the correct location")
            raise

print("\nAll imports successful!")
print("Using modern LangChain patterns: create_agent with @tool decorator")

x .
x ./usr
x ./usr/local
x ./usr/local/bin
x ./usr/local/bin/pandoc
x ./usr/local/bin/._pandoc
x ./usr/local/bin/pandoc-lua
x ./usr/local/bin/._pandoc-lua
x ./usr/local/bin/pandoc-server
x ./usr/local/bin/._pandoc-server
x ./usr/local/._bin
x ./usr/local/share
x ./usr/local/share/man
x ./usr/local/share/man/man1
x ./usr/local/share/man/man1/pandoc-lua.1
x ./usr/local/share/man/man1/._pandoc-lua.1
x ./usr/local/share/man/man1/pandoc-server.1
x ./usr/local/share/man/man1/._pandoc-server.1
x ./usr/local/share/man/man1/pandoc.1
x ./usr/local/share/man/man1/._pandoc.1
x ./usr/local/share/man/._man1
x ./usr/local/share/._man
x ./usr/local/._share
x ./usr/._local
x ./._usr


Loading .env from: /Users/kaliml/Cursor Projects/rag-qa-system/.env
OpenAI API key loaded successfully!
API key validated successfully!

All imports successful!
Using modern LangChain patterns: create_agent with @tool decorator


## Step 1: Load

- Use a LangChain document loader appropriate for your data type
- Print: number of documents loaded, sample of first document content

In [7]:

# Load website
web_loader = WebBaseLoader(
    "https://brokenscience.org/what-is-fitness-crossfit/",
    bs_kwargs={
        "parse_only": SoupStrainer(
            class_=["post-content"]  # only pull article content, not website navigation, etc.
        )
    }
)

#Load PDF
pdf_loader = PyPDFLoader("docs/safety_and_intensity_crossfit.pdf")

# Load ODT
odt_loader = UnstructuredODTLoader("docs/The_Foundation_Of_CrossFit_ Moving_With_Proper_Mechanics.odt")

# Combine all documents into one list
documents = (
    web_loader.load() +
    pdf_loader.load() +
    odt_loader.load()
    
)

# Trim the bottom noise from the webpage
cutoff_phrase = "This article, by BSI’s co-founder"
idx = documents[0].page_content.find(cutoff_phrase)
if idx != -1:
    documents[0].page_content = documents[0].page_content[:idx]

#Display document previews for review
print("-" * 60)
print(f"Loaded {len(documents)} document(s)")

for index, document in enumerate(documents):
    print(f"Document #", index+1)
    print(f"Verify start of document looks clean:")
    print("-" * 3)
    print(document.page_content[:300])
    print("-" * 3)
    print(f"\nVerify end of document looks clean:")
    print("-" * 3)
    print(document.page_content[-300:])
    print("-" * 3)
    print(f"\nDocument metadata: {document.metadata}")
    print("-" * 80)





------------------------------------------------------------
Loaded 5 document(s)
Document # 1
Verify start of document looks clean:
---

What Is Fitness and Who Is Fit?
In 1997, Outside Magazine crowned triathlete Mark Allen “the fittest man on Earth.” Let us just assume for a moment that this famous six-time winner of the IronMan Triathlon is the fittest of the fit. Then what title do we bestow on the decathlete Simon Poelman, who 
---

Verify end of document looks clean:
---
neffective. The need for specificity is nearly completely met by regular practice and training within the sport, not in the strength-and-conditioning environment. Our terrorist hunters, skiers, mountain bikers and housewives have found their best fitness from the same regimen.

Cover image: Dave Re

---

Document metadata: {'source': 'https://brokenscience.org/what-is-fitness-crossfit/'}
--------------------------------------------------------------------------------
Document # 2
Verify start of document looks c

## Step 2: Chunk

- Use RecursiveCharacterTextSplitter
- Print: total chunks created, character count of smallest and largest chunk
- Experiment: Try at least 2 different chunk_size values (e.g., 500 vs 1000) and note what changes

In [8]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Size of each chunk
    chunk_overlap=100,   # Overlap between chunks (important!)
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to split at these boundaries
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"\nChunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in chunks)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in chunks)} characters")

Split into 87 chunks

Chunk statistics:
  Average chunk size: 542 characters
  Min chunk size: 5 characters
  Max chunk size: 800 characters


## Step 3: Embed + Store

- Embed chunks using an embedding model of your choice
- Store in ChromaDB (or another vector DB if you're feeling adventurous)

In [9]:
import shutil

# Explicitly release the old connection first
try:
    del vectorstore
except NameError:
    pass  # No existing vectorstore, that's fine

# Now clear the folder
db_path = "./rag_vector_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("Cleared existing vector database")

# Create fresh
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print("Created fresh vector database!")
print(f"Stored {len(chunks)} document chunks")

# Create retrievers + ensemble (BM25 + vector similarity)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
bm25_retriever = BM25Retriever.from_documents(chunks, k=3)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3],
)

print("Created ensemble retriever (vector + BM25)")


Created fresh vector database!
Stored 87 document chunks
Created ensemble retriever (vector + BM25)


# Step 4: Test Retrieval (before wiring up the LLM!)

- Run 3 test queries using similarity_search
- For each query, print the top 3 retrieved chunks
- Annotate: Are the retrieved chunks actually relevant? Yes/No and why?

In [ ]:
#Retrieval Test Question setup
questions = [
    "What is a hopper used for?",
    "What is the risk of squatting with a rounded back?",
    "What is the coach responsible for?",
]

for question in questions:
    results = ensemble_retriever.invoke(question)[:2]

    print("-" * 80)
    print(f"\nTEST QUERY: '{question}'")
    print(f"\nRETRIEVED {len(results)} RELEVANT CHUNKS:")
    print("-" * 40)
    for i, doc in enumerate(results, 1):
        print(f"\n{i}. {doc.page_content[:200]}...")
        print(f"   METADATA: {doc.metadata}")
        print("-" * 15)


--------------------------------------------------------------------------------

TEST QUERY: 'What are arguably the two most common faults in fitness training?'

RETRIEVED 2 RELEVANT CHUNKS:
----------------------------------------

1. Total fitness, the fitness that CrossFit promotes and develops, requires competency and training in each of these three pathways or engines. Balancing the effects of these three pathways largely deter...
   METADATA: {'source': 'https://brokenscience.org/what-is-fitness-crossfit/'}
---------------

2. We have not mentioned here our penchant for jumping, kettlebells, odd-object lifting and obstacle-course work. The recurring theme of functionality and variety clearly suggests the need and validity f...
   METADATA: {'source': 'https://brokenscience.org/what-is-fitness-crossfit/'}
---------------
--------------------------------------------------------------------------------

TEST QUERY: 'What is the risk of squatting with a rounded back?'

RETRIEVED 2 R

## Step 5: Build the RAG Chain

- Wire up RetrievalQA with a custom prompt template
- Run the same 3 queries through the full chain
- Print the generated answers

In [ ]:
# Initialize LLM
model = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and cost-effective
    temperature=0  # Deterministic responses
)

# Create a retrieval tool using the modern @tool decorator pattern
# This follows the latest LangChain documentation pattern
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information from the document store to help answer a query."""
    retrieved_docs = ensemble_retriever.invoke(query)[:3]
    serialized = "\n\n".join(
        (f"SOURCE: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# Create the RAG agent using create_agent (modern pattern)
tools = [retrieve_context]
system_prompt = (
    "You are a helpful assistant that answers questions using a document retrieval tool. "
    "Always call the retrieval tool before answering — never rely on your own knowledge. "
    "Base your answer only on the retrieved content."
    "Answer only what is directly asked. Do not include related but unrequested information."
    "If the retrieved content does not contain enough information to answer the question, say so clearly — do not guess. "
    "Keep answers concise and specific. "
    "At the end of each answer, cite the source document and page number from the metadata."
)

rag_agent = create_agent(model, tools, system_prompt=system_prompt)

print("\nAGENT CONFIGURATION:")
print(f"  LLM: {model.model_name}")
print(f"  RETRIEVAL: Top 3 chunks")
print(f"  PATTERN: Agent with @tool decorator (latest LangChain docs)")

In [ ]:
#Use the questions defined previously

for question in questions:
    print(f"\n{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}\n")

    for event in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        last_message = event["messages"][-1]

        # Only print the final answer (skip tool calls and raw tool results)
        has_tool_calls = hasattr(last_message, "tool_calls") and last_message.tool_calls
        has_content = hasattr(last_message, "content") and last_message.content

        if has_content and not has_tool_calls:
            # Skip the echoed question (it's just a HumanMessage)
            if last_message.__class__.__name__ != "HumanMessage":
                print(f"{last_message.content}\n")

    print("-" * 70)

## Step 6: Evaluate (the part most people skip)

- Create a mini eval set: 5 questions where you know the correct answer
- Run all 5 through your pipeline
- For each, record:

    ✅ or ❌ Did it retrieve the right chunks?
    ✅ or ❌ Is the answer grounded in the context (not hallucinated)?
    ✅ or ❌ Is the answer actually correct?
- Calculate your accuracy: X/5 retrieval, X/5 faithfulness, X/5 correctness

In [ ]:
eval_questions = [
    "How do I achieve sustainable intensity?",
    "Why are warm-up routines important?",
    "What needs to happen before increasing load?",
    "Why would an athlete's intensity need to change?",
    "Who is responsible for managing threshold training?"
]

# Expected answers (for manual scoring)
expected_answers = [
    "Safety is the Foundation for Sustainable Intensity",
    "Warm-ups prepare the body for upcoming demands, allow coaches time for skill refinement and load management, and lay the groundwork for effective scaling. ... A key component of a session is a time to check in or assess your athlete’s readiness ... This can be achieved ... during the general and specific warm-up.",
    "Coaches guide athletes on a path to develop correct technique. After mechanics and consistency are established, we can then start to focus on increasing intensity",
    "An athlete's relative intensity may change frequently. For example, an athlete might have a poor night's sleep or be in a period of high stress ... intensity is relative to the individual’s physical and psychological tolerance ... / ",
    "The coach is responsible for managing this for every workout, for every athlete."
]

results = []

for i, question in enumerate(eval_questions):
    print(f"\n{'='*70}")
    print(f"❓ Question {i+1}: {question}")
    print(f"{'='*70}\n")

    final_answer = None

    for event in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        last_message = event["messages"][-1]
        has_tool_calls = hasattr(last_message, "tool_calls") and last_message.tool_calls
        has_content = hasattr(last_message, "content") and last_message.content
        is_human = last_message.__class__.__name__ == "HumanMessage"

        if has_content and not has_tool_calls and not is_human:
            final_answer = last_message.content

    print(f"💬 Answer:\n{final_answer}\n")
    print(f"📋 Expected:\n{expected_answers[i]}\n")



# Manual Scoring

| Q  | Retrieved | Grounded | Correct |
|----|-----------|----------|---------|
| 1  | Y         | Y        | Y       |
| 2  | Y         | Y        | N       |
| 3  | Y         | Y        | Y       |
| 4  | Y         | Y        | N       |
| 5  | Y         | Y        | Y       |
